In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType

In [0]:

df_ocorrencias_silver = spark.table("projeto_seguranca_rj.bronze.ocorrencias_municipio")

In [0]:
total = df_ocorrencias_silver.count()

sem_duplicatas = df_ocorrencias_silver.dropDuplicates().count()

print("Total:", total)
print("Após remover duplicatas:", sem_duplicatas)
print("Duplicatas:", total - sem_duplicatas)

In [0]:
duplicados_chave = (
    df_ocorrencias_silver
    .groupBy(
        "fmun_cod",
        "ano",
        "mes"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

qtd_duplicados = duplicados_chave.count()

print(
    "Chaves duplicadas:",
    qtd_duplicados
)

if qtd_duplicados > 0:
    display(duplicados_chave)

    raise ValueError(
        "Foram encontradas chaves duplicadas em "
        "ocorrencias_municipio."
    )

print("Unicidade da chave validada com sucesso.")

In [0]:
display(df_ocorrencias_silver.limit(5))


In [0]:
df_ocorrencias_silver = (
    df_ocorrencias_silver
    .withColumn(
        "data_referencia",
        F.to_date(
            F.concat_ws(
                "-",
                F.col("ano"),
                F.lpad(F.col("mes"),2,"0"),
                F.lit("01")
            )
        )
    )
)

In [0]:
datas_invalidas = (
    df_ocorrencias_silver
    .filter(
        F.col("data_referencia").isNull()
    )
    .count()
)

if datas_invalidas > 0:
    raise ValueError(
        f"Foram encontradas {datas_invalidas} "
        "datas de referência inválidas."
    )

print("data_referencia validada com sucesso.")

In [0]:
df_ocorrencias_silver.select(
    [
        F.count(
            F.when(
                F.col(c).isNull(),
                c
            )
        ).alias(c)
        for c in df_ocorrencias_silver.columns
    ]
).show()

In [0]:
colunas_nulas = [
    "feminicidio",
    "tentativa_feminicidio"
]

for coluna in colunas_nulas:
    df_ocorrencias_silver = (
        df_ocorrencias_silver
        .withColumn(
            coluna,
            F.coalesce(
                F.col(coluna),
                F.lit(0)
            )
        )
    )

In [0]:
nulos_restantes = (
    df_ocorrencias_silver
    .select(
        [
            F.count(
                F.when(
                    F.col(c).isNull(),
                    c
                )
            ).alias(c)
            for c in colunas_nulas
        ]
    )
    .collect()[0]
    .asDict()
)

problemas_nulos = {
    coluna: quantidade
    for coluna, quantidade in nulos_restantes.items()
    if quantidade > 0
}

if problemas_nulos:
    raise ValueError(
        f"Ainda existem valores nulos: "
        f"{problemas_nulos}"
    )

print(
    "Tratamento de feminicidio e "
    "tentativa_feminicidio validado com sucesso."
)

In [0]:
nulos_chave = (
    df_ocorrencias_silver
    .filter(
        F.col("fmun_cod").isNull()
        | F.col("fmun").isNull()
        | F.col("ano").isNull()
        | F.col("mes").isNull()
    )
    .count()
)

if nulos_chave > 0:
    raise ValueError(
        f"Foram encontrados {nulos_chave} registros "
        "com valores nulos na chave da tabela."
    )

print("Campos obrigatórios da chave validados com sucesso.")

In [0]:

df_ocorrencias_silver = (
    df_ocorrencias_silver
    .withColumn(
        "fmun_cod",
        F.col("fmun_cod").cast(StringType())
    )
)

In [0]:
df_ocorrencias_silver.printSchema()

In [0]:
display(
    df_ocorrencias_silver
    .select(
        "fmun_cod",
        "fmun",
        "ano",
        "mes",
        "data_referencia",
        "feminicidio",
        "tentativa_feminicidio"
    )
    .limit(10)
)

In [0]:
(
    df_ocorrencias_silver.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "projeto_seguranca_rj.silver.ocorrencias_municipio"
    )
)

print(
    "Tabela projeto_seguranca_rj.silver.ocorrencias_municipio "
    "salva com sucesso."
)

In [0]:
linhas_dataframe = df_ocorrencias_silver.count()

linhas_tabela = (
    spark.table(
        "projeto_seguranca_rj.silver.ocorrencias_municipio"
    )
    .count()
)

print(
    f"Silver ocorrências: DataFrame={linhas_dataframe} | "
    f"Tabela={linhas_tabela}"
)

if linhas_dataframe != linhas_tabela:
    raise ValueError(
        "Divergência entre o DataFrame tratado "
        "e a tabela Silver persistida."
    )

print(
    "Persistência da Silver ocorrências "
    "validada com sucesso."
)